# Avance 5 · Los ensambles de la permeabilidad con huellas

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Compañero del [05](05_Modelo_Final.ipynb) y del [03c](03c_Permeabilidad_huellas.ipynb). Responde una sola pregunta, la que ninguno de los dos contesta: **¿un ensamble saca de las huellas moleculares algo que un modelo individual no saca?**

El [03c](03c_Permeabilidad_huellas.ipynb) midió cinco representaciones sobre los seis modelos **individuales** de la fase 1 y ninguna mejoró a nadie, que es lo que dejó escrito la decisión [1.3](../docs/decisiones.md#d1-3): las huellas entran en la fase 2 y no en la fase 1. El [05](05_Modelo_Final.ipynb) midió los cinco **ensambles**, pero solo con descriptores. El cruce —ensambles **y** huellas— no estaba medido, y la razón para medirlo no es la esperanza de que gane: un ensamble combina modelos que se equivocan distinto, y una representación con 2,048 columnas nuevas podría separarlos más de lo que separa a cada uno por su cuenta. Si no pasa, eso **también** es el resultado, y conviene tenerlo medido en vez de supuesto.

Este cuaderno lleva sufijo de letra, así que **no es un entregable**: es la evidencia detrás de una frase del 05. No elige modelo final, no dibuja los cuatro gráficos y **no toca la partición de prueba**, que se lee una sola vez y se lee en el 05.

## 1. Preparación

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"
DATOS = RAIZ / "notebooks" / "datos"

from src.evaluation.cv import evaluar
from src.features.huellas import AgregarHuellas, agrupar_iguales, identificadores_morgan
from src.models import cargar, seleccion
from src.models.ensambles_clasificacion import CanalizacionConGrupos, ensambles

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

sistema  macOS-27.0.1-arm64-arm-64bit
python   3.12.12
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1
scipy    1.18.1


### Las huellas, y por qué envolver un ensamble no es gratis

Una huella **Sort & Slice** es una tabla de 0 y 1: una columna por subestructura —el entorno de radio 2 alrededor de un átomo, el mismo que ECFP4— y un 1 si la molécula la tiene. De todas las que aparecen se conservan las **2,048 más frecuentes**. A diferencia de una huella plegada por hash, cada columna es **una** subestructura, y por eso una columna con peso alto se puede traducir a un fragmento concreto; es el argumento de explicabilidad que pide la [C.2](../docs/decisiones.md#dc-2).

**El vocabulario se aprende en cada ajuste, con las filas de entrenamiento del pliegue.** Qué subestructuras son «las más frecuentes» depende de con qué filas se cuenten, así que calcularlo una vez sobre todo el conjunto dejaría que la validación influyera en qué columnas existen. `AgregarHuellas` va dentro de la canalización por eso, y `src/features/huellas.py` lo explica con detalle.

Hay un matiz que este cuaderno sí introduce y conviene decirlo: el `stacking` parte por dentro, y sus pliegues internos **heredan** el vocabulario elegido con todas las filas de entrenamiento del pliegue externo, no vuelven a elegirlo. No se mira la etiqueta para elegirlo —`AgregarHuellas.fit` solo cuenta frecuencias—, así que no es una fuga de la etiqueta, pero la frase honesta es que el meta-modelo ve columnas escogidas con filas que sus bases usaron para validar. La evaluación externa no se ve afectada.

Y envolver un ensamble en una canalización rompería algo que no se ve:

```python
class CanalizacionConGrupos(Pipeline):
    @property
    def necesita_grupos(self) -> bool:
        return bool(getattr(self.steps[-1][1], "necesita_grupos", False))

    def fit(self, X, y=None, grupos=None, **kw):
        if grupos is not None:
            kw[f"{self.steps[-1][0]}__grupos"] = grupos
        return super().fit(X, y, **kw)
```

El arnés mira `necesita_grupos` **en el objeto que recibe** y, si está puesto, llama `fit(X, y, grupos=...)`. Un `Pipeline` corriente no declara ese atributo ni acepta ese argumento, así que ponerle un `AgregarHuellas` delante al `stacking` con `make_pipeline` haría que el arnés dejara de pasarle los grupos: `StackingAgrupado` reventaría —porque se niega a ajustar sin ellos, y eso es a propósito— o, si no se negara, partiría al azar por dentro y la fuga no aparecería en ninguna métrica. La subclase hereda la exigencia del último paso y le reenvía el argumento.

### Los datos

Entrenamiento y validación de la fase 1, con la prueba reservada fuera. Los mismos compuestos, los mismos grupos por andamio y esqueleto de conectividad, las mismas tres semillas y los mismos cinco pliegues que el 03, el 04 y el 05, así que las filas de este cuaderno se pueden poner al lado de las de aquellos.

Cada compuesto lleva su **origen** —natural exacto, análogo estereoquímico o sin equivalente natural—, que es el desglose de la decisión [1.6](../docs/decisiones.md#d1-6): un promedio de todos no dice cómo le va al modelo con los productos naturales, que son los que el proyecto quiere priorizar.

In [2]:
t = cargar("permeabilidad", estratos="origen")
y, g, origen = t.y, t.grupos, t.estratos
X_control = t.X                                  # la matriz de descriptores del 03, el 04 y el 05
smiles = t.meta["SMILES"].to_numpy()
assert len(smiles) == len(y) == X_control.shape[0]

N_CONTROL = X_control.shape[1]
print(f"{len(y):,} compuestos · {y.mean():.1%} BBB+ · {len(np.unique(g)):,} grupos · "
      f"{N_CONTROL} descriptores")
print(pd.Series(origen).value_counts().to_string())

6,306 compuestos · 65.0% BBB+ · 1,048 grupos · 151 descriptores
natural exacto             2920
sin equivalente natural    2618
análogo estereoquímico      768


### Las huellas de los compuestos

Se leen los SMILES una sola vez y se arma una matriz dispersa con una columna por subestructura. **Todavía no se elige ningún vocabulario**: eso pasa dentro de cada ajuste. Antes se agrupan las subestructuras que valen lo mismo en todos los compuestos —porque siempre aparecen juntas, como el `S` y el `O=S` de un sulfonilo—: se queda una sola columna y se guarda el nombre de todas las que representa. Eso no mira la etiqueta y lo idéntico en todas las filas lo es en cualquier subconjunto, así que hacerlo una vez no filtra nada.

Las columnas de descriptores van primero y las huellas después, en una sola matriz dispersa sin densificar.

#### Las huellas, por dentro

Es el sujeto de este cuaderno, así que su código va aquí y no en otro. `AgregarHuellas` es un transformador: **aprende el vocabulario de Sort & Slice en `fit`**, o sea dentro del pliegue de entrenamiento, y lo aplica en `transform`. Aprenderlo antes de partir sería mirar los compuestos de validación para decidir qué columnas existen.

```python
# src/features/huellas.py
class AgregarHuellas(BaseEstimator, TransformerMixin):
    """Entrada mixta → matriz densa con las columnas `densas` y las `n_bits` huellas más frecuentes.

    `X` es una matriz dispersa cuyas primeras columnas son las densas (descriptores y diana) y el
    resto es la salida de `identificadores_morgan`. En `fit` cuenta cuántas filas de entrenamiento
    tiene cada subestructura y se queda con las `n_bits` más frecuentes (los empates se resuelven por
    la posición de la columna, que sigue el orden del identificador, para que sea reproducible).
    `transform` devuelve las columnas densas indicadas y esas huellas, en `float32`.

    Con `densas=()` solo salen las huellas.
    """

    def __init__(self, n_densas: int, densas: tuple[int, ...] | None = None, n_bits: int = 2048):
        self.n_densas = n_densas
        self.densas = densas
        self.n_bits = n_bits

    def fit(self, X, y=None):
        huellas = sparse.csr_matrix(X)[:, self.n_densas:]
        frecuencia = np.asarray(huellas.sum(axis=0)).ravel()
        orden = np.lexsort((np.arange(len(frecuencia)), -frecuencia.astype(np.int64)))
        usadas = orden[:self.n_bits]
        self.columnas_ = usadas[frecuencia[usadas] > 0]
        return self

    def transform(self, X):
        X = sparse.csr_matrix(X)
        densas = np.arange(self.n_densas) if self.densas is None else np.asarray(self.densas, int)
        partes = [X[:, densas].toarray().astype(np.float32),
                  X[:, self.n_densas:][:, self.columnas_].toarray().astype(np.float32)]
        return np.hstack(partes)
```

In [3]:
t0 = time.perf_counter()
huellas, identificadores = identificadores_morgan(smiles)
n_distintas = huellas.shape[1]
huellas, identificadores, equivalentes = agrupar_iguales(huellas, identificadores)
n_agrupadas = sum(len(e) > 1 for e in equivalentes)
X_mixta = sparse.hstack([sparse.csr_matrix(X_control), huellas]).tocsr()
print(f"{n_distintas:,} subestructuras distintas en {huellas.shape[0]:,} compuestos; "
      f"{n_distintas - huellas.shape[1]:,} valían lo mismo que otra y quedan {huellas.shape[1]:,} columnas "
      f"({n_agrupadas} de ellas representan a más de una) · {time.perf_counter() - t0:.1f} s")
print(f"X_mixta {X_mixta.shape} · cada modelo verá {N_CONTROL} + 2,048 columnas")

13,817 subestructuras distintas en 6,306 compuestos; 7,612 valían lo mismo que otra y quedan 6,205 columnas (2287 de ellas representan a más de una) · 0.5 s
X_mixta (6306, 6356) · cada modelo verá 151 + 2,048 columnas


## 2. Dos brazos, los mismos nueve modelos

Los nueve son los cuatro individuales del 04 que el 05 usó como referencia y como bases —el SVM RBF, la logística, la red de una capa y los vecinos— y los cinco ensambles del 05, **sin tocar su definición**: vienen de `src/models/ensambles_clasificacion.py` tal cual, para que su fila con descriptores reproduzca la que el 05 registró.

**Dos brazos y no cinco.** El 03c ya midió `huellas`, `seleccionados` y `depurados` sobre los individuales, y ninguno mejoró a nadie; repetirlos aquí contestaría una pregunta que ya tiene respuesta. Los dos que quedan son los que contestan esta: `descriptores`, que es el control y además reproduce el 05, y `descriptores+huellas`. Añadir un tercer brazo es una línea en `BRAZOS`.

**Los afinados quedan fuera.** El 05 midió el bosque y el boosting con búsqueda de rejilla y ninguno de los dos pagó su costo. Este cuaderno no pregunta por el afinado.

### Lo que se fija antes de ver los resultados

- **Las métricas principales son el MCC y el AUPRC de la clase minoritaria** ([C.1](../docs/decisiones.md#dc-1)); el AUROC se reporta como tercera lectura y no decide.
- **Las huellas mejoran a un modelo solo si** la diferencia media contra su propio brazo `descriptores` supera el umbral **y** es positiva en al menos **12 de las 15** evaluaciones, en las dos métricas principales a la vez. Las diferencias se calculan **en la misma evaluación** —misma semilla, mismo pliegue, mismo modelo—, no entre promedios.
- **El umbral de cada métrica es la mayor desviación que esa métrica muestra entre las 15 evaluaciones del control**, calculado solo con el brazo `descriptores` antes de mirar el otro. Es la misma regla con la que se leyeron el 03c, el 04c y el 05 ([1.3](../docs/decisiones.md#d1-3)), y se usa aquí por consistencia.
- Tres advertencias que acompañan a la regla siempre que se cite: **no está pre-registrada**, las 15 evaluaciones **no son independientes** porque las semillas comparten filas, y en la fase 1 el umbral sale **conservador** —la dispersión del MCC entre pliegues agrupados es grande—, así que se reporta como tal en vez de bajarlo para que algo pase.

#### Los nueve modelos, tal como los definen el catálogo y los ensambles

Ninguno de los nueve se define en este cuaderno, y es a propósito: su brazo de control tiene que **reproducir la corrida registrada del 05**, así que los individuales salen del catálogo y los ensambles de `ensambles`, los mismos objetos. `tests/test_05c_codigo.py` ata las dos cosas: que aquí no se redefina ninguno, y que este código mostrado sea el de `src/`.

```python
# src/models/alternativos.py
def _con_escala(modelo) -> Pipeline:
    return Pipeline([("escala", MinMaxScaler()), ("modelo", modelo)])


def catalogo(semilla: int = 0) -> dict[str, Pipeline]:
    """Las ocho familias, listas para `evaluar`. Elegir seis es del equipo."""
    return {
        "logistica": _con_escala(
            LogisticRegression(max_iter=3000, random_state=semilla)),
        "lda": _con_escala(LinearDiscriminantAnalysis()),
        "arbol": Pipeline([
            ("modelo", DecisionTreeClassifier(min_samples_leaf=5,
                                              random_state=semilla))]),
        "knn": _con_escala(KNeighborsClassifier(n_neighbors=15)),
        "bayes": _con_escala(GaussianNB()),
        # SVC no da `predict_proba` por sí solo y sin ella el arnés no puede
        # medir AUROC ni AUPRC. La calibración se la agrega por fuera: en
        # scikit-learn 1.9 `probability=True` quedó deprecado a favor de esto, y
        # con `ensemble=False` ajusta un solo modelo en vez de uno por pliegue.
        # El escalador va DENTRO de la calibración (E17, #262): fuera, la calibración
        # ajustaba Platt sobre datos que el escalador ya había visto. El efecto medido
        # es nulo (MCC 0.7015 contra 0.7017; AUROC y AUPRC idénticos), pero la forma
        # correcta no depende de que el efecto sea pequeño.
        "svm_rbf": Pipeline([("modelo", CalibratedClassifierCV(
            Pipeline([("escala", MinMaxScaler()), ("svc", SVC(random_state=semilla))]),
            ensemble=False))]),
        # `modified_huber` es la pérdida que sí expone `predict_proba` en SGD.
        "svm_sgd": _con_escala(
            SGDClassifier(loss="modified_huber", max_iter=3000,
                          random_state=semilla)),
        "mlp": _con_escala(
            MLPClassifier(hidden_layer_sizes=(64,), max_iter=600,
                          random_state=semilla)),
    }


def seleccion(claves: tuple[str, ...], semilla: int = 0) -> dict[str, Pipeline]:
    """Las familias que el equipo haya elegido, por clave, fallando si no existen."""
    disponible = catalogo(semilla)
    faltan = [c for c in claves if c not in disponible]
    if faltan:
        raise ValueError(f"no están en el catálogo: {faltan}; hay {FAMILIAS}")
    return {c: disponible[c] for c in claves}
```

```python
# src/models/ensambles_clasificacion.py
def ensambles(base: Sequence[str], semilla: int = 0) -> dict[str, BaseEstimator]:
    """Los cinco ensambles, con `base` como los individuales de la votación y el stacking.

    `base` son claves del catálogo de `alternativos.py`; el bagging copia el SVM RBF
    porque es el mejor individual del Avance 4. El bosque y el boosting son los del
    baseline del 03, sin cambios, para que su fila se pueda contrastar con la que ya
    está registrada.
    """
    individuales = seleccion(tuple(base), semilla)
    referencia = canalizaciones(semilla)
    # El bosque del baseline usa todos los núcleos; aquí uno solo, como los demás, para que
    # la columna de tiempos compare modelos y no paralelismo. No cambia sus predicciones.
    referencia["bosque"].set_params(modelo__n_jobs=1)
    svm = seleccion(("svm_rbf",), semilla)["svm_rbf"]
    return {
        "bosque": referencia["bosque"],
        "boosting": referencia["boosting"],
        "bagging_svm": Pipeline([
            ("modelo", BaggingClassifier(svm, n_estimators=10, random_state=semilla))]),
        "votacion": Pipeline([
            ("modelo", VotingClassifier(list(individuales.items()), voting="soft"))]),
        "stacking": StackingAgrupado(list(individuales.items()), semilla=semilla),
    }
```

In [4]:
SEMILLAS = (0, 1, 2)
N_BITS = 2048                                    # tamaño del vocabulario de Sort & Slice
MINIMO_PLIEGUES = 12                             # de 15 evaluaciones, en cuántas tiene que mejorar
MUESTRA_ENTRENAMIENTO = 3000                     # filas de entrenamiento para medir la brecha
PLIEGUES_BRECHA = 2                              # la brecha es un diagnóstico, no una cifra que decida
BASE = ("svm_rbf", "logistica", "mlp", "knn")     # los cuatro del 04, bases de los heterogéneos
MEJOR_INDIVIDUAL = "svm_rbf"                      # el que eligió el 04
PRINCIPALES = ("mcc", "auprc_minoritaria")        # las que deciden; el AUROC no

individuales = seleccion(BASE)
MODELOS = {**individuales, **ensambles(BASE)}
ORDEN_MODELOS = list(BASE) + ["bosque", "boosting", "bagging_svm", "votacion", "stacking"]
assert set(MODELOS) == set(ORDEN_MODELOS)


def con_huellas(m):
    """El mismo modelo, con el vocabulario aprendido dentro del ajuste y los grupos intactos."""
    return CanalizacionConGrupos([("huellas", AgregarHuellas(N_CONTROL, n_bits=N_BITS)),
                                  ("modelo", m)])


BRAZOS = {
    "descriptores":         (X_control, lambda m: m),
    "descriptores+huellas": (X_mixta,   con_huellas),
}
orden_brazos = list(BRAZOS)
print(f"{len(MODELOS)} modelos × {len(BRAZOS)} brazos × {len(SEMILLAS)} semillas × 5 pliegues")

9 modelos × 2 brazos × 3 semillas × 5 pliegues


In [5]:
partes = []
inicio = time.perf_counter()
for brazo, (matriz, armar) in BRAZOS.items():
    pipes = {nombre: armar(m) for nombre, m in MODELOS.items()}
    r, _ = evaluar(pipes, matriz, y, g, estratos=origen, semillas=SEMILLAS, verboso=False)
    partes.append(r.assign(brazo=brazo))
    print(f"{brazo:<22} {(time.perf_counter() - inicio) / 60:6.1f} min", flush=True)
res = pd.concat(partes, ignore_index=True)
print(f"{len(res):,} filas")

descriptores              8.8 min


descriptores+huellas     89.9 min


1,080 filas


## 3. Resultados

### El control: el brazo `descriptores` repite el 05

Antes de leer nada sobre las huellas hay que comprobar que el brazo de control reproduce lo que ya está registrado. Se compara contra la corrida del 05 **de esta misma máquina** —`2026-10-29-artemio-1.5-ensambles-permeabilidad`—, no contra la de Linux que trae este PR: la matriz de descriptores de la fase 1 no es idéntica entre las máquinas del equipo ([#166](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/166)) y comparar contra otra máquina mezclaría esa diferencia con cualquier otra.

Si una fila no reproduce, el problema está en este cuaderno y no en las huellas.

In [6]:
REGISTRO = RAIZ / "experimentos" / "2026-10-29-artemio-1.5-ensambles-permeabilidad" / "metricas.json"
registrado = pd.DataFrame(json.loads(REGISTRO.read_text())["comparativa"]).T
aqui = (res[(res["brazo"] == "descriptores") & (res["estrato"] == "todos")]
        .groupby("modelo")[["mcc", "auprc_minoritaria", "auroc"]].mean())
control = pd.concat({"05 (registrado)": registrado[["mcc", "auprc_minoritaria", "auroc"]],
                     "05c (esta corrida)": aqui}, axis=1).loc[ORDEN_MODELOS]
for m in ("mcc", "auprc_minoritaria", "auroc"):
    control[("diferencia", m)] = control[("05c (esta corrida)", m)] - control[("05 (registrado)", m)]
display(control.round(4))
print(f"filas que reproducen a tres decimales: "
      f"{int((control['diferencia'].abs() < 5e-4).all(axis=1).sum())} de {len(control)}")

05 (registrado)                           05c (esta corrida)  \
                        mcc auprc_minoritaria   auroc                mcc   
svm_rbf              0.7017            0.8842  0.9190             0.7017   
logistica            0.6866            0.8724  0.9133             0.6866   
mlp                  0.6743            0.8640  0.9110             0.6743   
knn                  0.6692            0.8633  0.9141             0.6692   
bosque               0.6996            0.8950  0.9329             0.6996   
boosting             0.7019            0.8843  0.9302             0.7019   
bagging_svm          0.6948            0.8862  0.9213             0.6948   
votacion             0.7092            0.8949  0.9329             0.7092   
stacking             0.7109            0.8935  0.9318             0.7109   

                                      diferencia                          
            auprc_minoritaria   auroc        mcc auprc_minoritaria auroc  
svm_rbf                0.8842  0.9190       -0.0              -0.0   0.0  
logistica              0.8724  0.9133        0.0               0.0   0.0  
mlp                    0.8640  0.9110       -0.0               0.0  -0.0  
knn                    0.8633  0.9141        0.0              -0.0   0.0  
bosque                 0.8950  0.9329        0.0              -0.0  -0.0  
boosting               0.8843  0.9302        0.0              -0.0  -0.0  
bagging_svm            0.8862  0.9213       -0.0              -0.0  -0.0  
votacion               0.8949  0.9329       -0.0               0.0   0.0  
stacking               0.8935  0.9318        0.0               0.0   0.0

filas que reproducen a tres decimales: 9 de 9


### Los dos brazos, modelo por modelo

In [7]:
def tabla(r, estrato="todos"):
    """Media por brazo y modelo de cada métrica, con la dispersión del MCC entre evaluaciones."""
    r = r[r["estrato"] == estrato]
    t = r.groupby(["brazo", "modelo"])[["mcc", "auprc_minoritaria", "auroc", "segundos_ajuste"]].mean()
    t.insert(1, "desv_mcc", r.groupby(["brazo", "modelo"])["mcc"].std())
    return t.round(4)


tabla_todos = tabla(res)
for m in ("mcc", "auprc_minoritaria", "auroc"):
    print(m)
    display(tabla_todos[m].unstack("brazo")[orden_brazos].loc[ORDEN_MODELOS])

mcc


brazo,descriptores,descriptores+huellas
modelo,,
svm_rbf,0.7017,0.6940
logistica,0.6866,0.6629
mlp,0.6743,0.6441
knn,0.6692,0.6505
bosque,0.6996,0.7039
boosting,0.7019,0.7087
bagging_svm,0.6948,0.6982
votacion,0.7092,0.6829
stacking,0.7109,0.6944


auprc_minoritaria


brazo,descriptores,descriptores+huellas
modelo,,
svm_rbf,0.8842,0.8742
logistica,0.8724,0.8534
mlp,0.8640,0.8281
knn,0.8633,0.8569
bosque,0.8950,0.8970
boosting,0.8843,0.8850
bagging_svm,0.8862,0.8820
votacion,0.8949,0.8820
stacking,0.8935,0.8872


auroc


brazo,descriptores,descriptores+huellas
modelo,,
svm_rbf,0.9190,0.9174
logistica,0.9133,0.9091
mlp,0.9110,0.8964
knn,0.9141,0.9085
bosque,0.9329,0.9357
boosting,0.9302,0.9317
bagging_svm,0.9213,0.9232
votacion,0.9329,0.9238
stacking,0.9318,0.9268


### ¿Mejoran las huellas? La comparación pliegue por pliegue

Cada modelo contra **sí mismo** con descriptores, en la misma semilla y el mismo pliegue. `mejor_en` cuenta en cuántas de las 15 gana; `cumple` aplica la regla a esa métrica y `mejora` exige las dos principales a la vez. Los umbrales se calculan primero, solo con el control.

In [8]:
control_todos = res[(res["brazo"] == "descriptores") & (res["estrato"] == "todos")]
UMBRALES = {m: round(float(control_todos.groupby("modelo")[m].std().max()), 3) for m in PRINCIPALES}
print("Umbrales: la mayor desviación de cada métrica del control entre sus 15 evaluaciones")
print(UMBRALES)


def comparacion(r, estrato="todos"):
    """Diferencia de cada brazo contra `descriptores`, evaluación por evaluación."""
    r = r[r["estrato"] == estrato]
    filas = []
    for metrica in ("mcc", "auprc_minoritaria", "auroc"):
        p = r.pivot_table(index=["modelo", "semilla", "pliegue"], columns="brazo", values=metrica)
        for brazo in orden_brazos[1:]:
            dif = (p[brazo] - p["descriptores"]).dropna()
            for modelo, v in dif.groupby("modelo"):
                filas.append({"brazo": brazo, "modelo": modelo, "metrica": metrica,
                              "evaluaciones": len(v), "dif_media": v.mean(), "dif_desv": v.std(),
                              "mejor_en": int((v > 0).sum())})
    c = pd.DataFrame(filas).set_index(["brazo", "modelo", "metrica"])
    c["cumple"] = (c["dif_media"].gt(c.index.get_level_values("metrica").map(UMBRALES).astype(float))
                   & c["mejor_en"].ge(MINIMO_PLIEGUES))
    mejora = c["cumple"].unstack("metrica")[list(PRINCIPALES)].all(axis=1).rename("mejora")
    return c.round(4), mejora


comparacion_todos, mejora_todos = comparacion(res)
display(comparacion_todos.unstack("metrica")[["dif_media", "mejor_en"]].round(3))
mejora_todos.unstack("brazo")[orden_brazos[1:]].loc[ORDEN_MODELOS]

Umbrales: la mayor desviación de cada métrica del control entre sus 15 evaluaciones
{'mcc': 0.081, 'auprc_minoritaria': 0.045}


dif_media                \
metrica                          auprc_minoritaria  auroc    mcc   
brazo                modelo                                        
descriptores+huellas bagging_svm            -0.004  0.002  0.003   
                     boosting                0.001  0.002  0.007   
                     bosque                  0.002  0.003  0.004   
                     knn                    -0.006 -0.006 -0.019   
                     logistica              -0.019 -0.004 -0.024   
                     mlp                    -0.036 -0.015 -0.030   
                     stacking               -0.006 -0.005 -0.017   
                     svm_rbf                -0.010 -0.002 -0.008   
                     votacion               -0.013 -0.009 -0.026   

                                          mejor_en            
metrica                          auprc_minoritaria auroc mcc  
brazo                modelo                                   
descriptores+huellas bagging_svm                 6     7   6  
                     boosting                    8     9  11  
                     bosque                      9    10   9  
                     knn                         5     7   5  
                     logistica                   3     7   7  
                     mlp                         2     3   5  
                     stacking                    5     5   4  
                     svm_rbf                     4     7   7  
                     votacion                    4     3   5

brazo,descriptores+huellas
modelo,
svm_rbf,False
logistica,False
mlp,False
knn,False
bosque,False
boosting,False
bagging_svm,False
votacion,False
stacking,False


### Con huellas, ¿algún ensamble supera al mejor individual?

La otra lectura, la del 05: dentro del brazo con huellas, cada ensamble contra el `svm_rbf` **medido en el mismo brazo y la misma evaluación**. Es la pregunta de la rúbrica —si el ensamble vale la pena— hecha sobre la representación nueva, y usa la misma regla con el umbral recalculado sobre el control de ese brazo.

In [9]:
def contra_individual(r, brazo, estrato="todos"):
    """Cada modelo contra el mejor individual del mismo brazo, evaluación por evaluación."""
    r = r[(r["brazo"] == brazo) & (r["estrato"] == estrato)]
    umbrales = {m: round(float(r[r["modelo"].eq(MEJOR_INDIVIDUAL)][m].std()), 3) for m in PRINCIPALES}
    filas = []
    for metrica in ("mcc", "auprc_minoritaria", "auroc"):
        p = r.pivot_table(index=["semilla", "pliegue"], columns="modelo", values=metrica)
        for modelo in [m for m in ORDEN_MODELOS if m != MEJOR_INDIVIDUAL]:
            dif = (p[modelo] - p[MEJOR_INDIVIDUAL]).dropna()
            filas.append({"modelo": modelo, "metrica": metrica, "evaluaciones": len(dif),
                          "dif_media": dif.mean(), "dif_desv": dif.std(),
                          "mejor_en": int((dif > 0).sum())})
    c = pd.DataFrame(filas).set_index(["modelo", "metrica"])
    c["cumple"] = (c["dif_media"].gt(c.index.get_level_values("metrica").map(umbrales).astype(float))
                   & c["mejor_en"].ge(MINIMO_PLIEGUES))
    mejora = c["cumple"].unstack("metrica")[list(PRINCIPALES)].all(axis=1).rename("supera")
    return umbrales, c.round(4), mejora


for brazo in orden_brazos:
    umbrales_b, contra, supera = contra_individual(res, brazo)
    print(f"{brazo} · umbrales del {MEJOR_INDIVIDUAL} en este brazo: {umbrales_b}")
    display(contra.unstack("metrica")[["dif_media", "mejor_en"]].round(3))
    display(supera.to_frame().T)

descriptores · umbrales del svm_rbf en este brazo: {'mcc': 0.073, 'auprc_minoritaria': 0.041}


dif_media                        mejor_en          
metrica     auprc_minoritaria  auroc    mcc auprc_minoritaria auroc mcc
modelo                                                                 
bagging_svm             0.002  0.002 -0.007                11    11   4
boosting                0.000  0.011  0.000                 8    12   8
bosque                  0.011  0.014 -0.002                12    14   8
knn                    -0.021 -0.005 -0.032                 2     7   2
logistica              -0.012 -0.006 -0.015                 4     4   3
mlp                    -0.020 -0.008 -0.027                 2     3   3
stacking                0.009  0.013  0.009                11    14  12
votacion                0.011  0.014  0.008                14    15   9

modelo,bagging_svm,boosting,bosque,knn,logistica,mlp,stacking,votacion
supera,False,False,False,False,False,False,False,False


descriptores+huellas · umbrales del svm_rbf en este brazo: {'mcc': 0.076, 'auprc_minoritaria': 0.039}


dif_media                        mejor_en          
metrica     auprc_minoritaria  auroc    mcc auprc_minoritaria auroc mcc
modelo                                                                 
bagging_svm             0.008  0.006  0.004                14    15   8
boosting                0.011  0.014  0.015                12    14  11
bosque                  0.023  0.018  0.010                14    15   8
knn                    -0.017 -0.009 -0.044                 4     5   2
logistica              -0.021 -0.008 -0.031                 1     3   4
mlp                    -0.046 -0.021 -0.050                 0     1   1
stacking                0.013  0.009  0.000                13    13   7
votacion                0.008  0.006 -0.011                12    12   6

modelo,bagging_svm,boosting,bosque,knn,logistica,mlp,stacking,votacion
supera,False,False,False,False,False,False,False,False


### Por origen

Los umbrales se calcularon con el agregado, así que estas filas son **exploratorias**: sirven para ver dónde cae la diferencia, no para declarar una mejora. Importa porque en la fase 2 la ganancia de las huellas se concentró en los pares que solo tiene ChEMBL y no en los productos naturales, y aquí hay que mirar si pasa lo mismo.

In [10]:
por_origen = list(pd.unique(res.loc[res["estrato"].ne("todos"), "estrato"]))
mcc_origen = pd.concat({e: tabla(res, e)["mcc"].unstack("brazo")[orden_brazos] for e in por_origen},
                       axis=1)
display(mcc_origen.loc[ORDEN_MODELOS].round(4))
pd.concat({e: comparacion(res, e)[0].query("metrica in @PRINCIPALES")[["dif_media", "mejor_en"]]
           for e in por_origen}, names=["origen"]).round(3)

sin equivalente natural                      natural exacto  \
brazo                  descriptores descriptores+huellas   descriptores   
modelo                                                                    
svm_rbf                      0.4989               0.5478         0.7182   
logistica                    0.5001               0.5083         0.6924   
mlp                          0.5406               0.4941         0.6703   
knn                          0.4762               0.5102         0.6935   
bosque                       0.5218               0.5257         0.7145   
boosting                     0.5209               0.5409         0.7190   
bagging_svm                  0.4958               0.5599         0.7092   
votacion                     0.5367               0.5437         0.7204   
stacking                     0.5527               0.5745         0.7194   

                                 análogo estereoquímico                       
brazo       descriptores+huellas           descriptores descriptores+huellas  
modelo                                                                        
svm_rbf                   0.7070                 0.6958               0.6134  
logistica                 0.6832                 0.6916               0.5991  
mlp                       0.6596                 0.6558               0.5834  
knn                       0.6606                 0.6187               0.5757  
bosque                    0.7208                 0.6660               0.6602  
boosting                  0.7253                 0.6830               0.6769  
bagging_svm               0.7087                 0.6911               0.6244  
votacion                  0.6961                 0.6900               0.5971  
stacking                  0.7013                 0.6841               0.6072

dif_media  \
origen                  brazo                modelo      metrica                        
sin equivalente natural descriptores+huellas bagging_svm mcc                    0.064   
                                             boosting    mcc                    0.020   
                                             bosque      mcc                    0.004   
                                             knn         mcc                    0.034   
                                             logistica   mcc                    0.008   
                                             mlp         mcc                   -0.046   
                                             stacking    mcc                    0.022   
                                             svm_rbf     mcc                    0.049   
                                             votacion    mcc                    0.007   
                                             bagging_svm auprc_minoritaria      0.012   
                                             boosting    auprc_minoritaria      0.010   
                                             bosque      auprc_minoritaria      0.011   
                                             knn         auprc_minoritaria      0.010   
                                             logistica   auprc_minoritaria     -0.031   
                                             mlp         auprc_minoritaria     -0.064   
                                             stacking    auprc_minoritaria     -0.019   
                                             svm_rbf     auprc_minoritaria     -0.006   
                                             votacion    auprc_minoritaria     -0.039   
natural exacto          descriptores+huellas bagging_svm mcc                   -0.000   
                                             boosting    mcc                    0.006   
                                             bosque      mcc                    0.006   
                                             knn         mcc                   -0.033   
                                             logistica   mcc                   -0.009   
                                             mlp         mcc                   -0.011   
                                             stacking    mcc                   -0.018   
                                             svm_rbf     mcc                   -0.011   
                                             votacion    mcc                   -0.024   
                                             bagging_svm auprc_minoritaria     -0.004   
                                             boosting    auprc_minoritaria      0.002   
                                             bosque      auprc_minoritaria      0.004   
                                             knn         auprc_minoritaria     -0.006   
                                             logistica   auprc_minoritaria     -0.008   
                                             mlp         auprc_minoritaria     -0.019   
                                             stacking    auprc_minoritaria      0.002   
                                             svm_rbf     auprc_minoritaria     -0.009   
                                             votacion    auprc_minoritaria     -0.002   
análogo estereoquímico  descriptores+huellas bagging_svm mcc                   -0.067   
                                             boosting    mcc                   -0.006   
                                             bosque      mcc                   -0.006   
                                             knn         mcc                   -0.043   
                                             logistica   mcc                   -0.092   
                                             mlp         mcc                   -0.072   
                                             stacking    mcc                   -0.077   
                                             svm_rbf     mcc           

### La brecha entre entrenamiento y validación

2,048 columnas nuevas sobre 6,306 compuestos invitan a memorizar, y el promedio no lo muestra. En la fase 2 la red con huellas llegó a 0.252 de brecha y fue el único modelo que empeoró. Se mide con el AUROC sobre una muestra de las filas de entrenamiento y sobre el pliegue de validación, con una semilla y los dos primeros pliegues, porque la brecha es un diagnóstico y no una cifra que decida: ninguna conclusión de este cuaderno se apoya en ella, y volver a ajustar los nueve modelos de los dos brazos en los cinco pliegues costaría más que la comparativa entera.

In [11]:
filas_brecha = []
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)
for k, (ajuste, val) in enumerate(list(cv.split(X_control, y, g))[:PLIEGUES_BRECHA], 1):
    muestra = np.random.default_rng(k).choice(ajuste, min(MUESTRA_ENTRENAMIENTO, len(ajuste)),
                                              replace=False)
    for brazo, (matriz, armar) in BRAZOS.items():
        for nombre, base in MODELOS.items():
            m = armar(base)
            if getattr(m, "necesita_grupos", False):
                m.fit(matriz[ajuste], y[ajuste], grupos=g[ajuste])
            else:
                m.fit(matriz[ajuste], y[ajuste])
            filas_brecha.append({
                "brazo": brazo, "modelo": nombre, "pliegue": k,
                "auroc_entrenamiento": roc_auc_score(y[muestra],
                                                     m.predict_proba(matriz[muestra])[:, 1]),
                "auroc_validacion": roc_auc_score(y[val], m.predict_proba(matriz[val])[:, 1])})
brecha = pd.DataFrame(filas_brecha).groupby(["brazo", "modelo"])[
    ["auroc_entrenamiento", "auroc_validacion"]].mean()
brecha["brecha"] = brecha["auroc_entrenamiento"] - brecha["auroc_validacion"]
brecha.round(4).unstack("brazo")["brecha"][orden_brazos].loc[ORDEN_MODELOS]

brazo,descriptores,descriptores+huellas
modelo,,
svm_rbf,0.0475,0.0737
logistica,0.0357,0.0966
mlp,0.0784,0.1223
knn,0.0492,0.0585
bosque,0.0700,0.0667
boosting,0.0733,0.0713
bagging_svm,0.0463,0.0682
votacion,0.0523,0.0784
stacking,0.0492,0.0644


In [12]:
# Registro en experimentos/ (decisión 1.3: qué representación reciben los modelos).
from src.tracking import Corrida

with Corrida("ensambles-permeabilidad-huellas", decision="1.3",
             params={"modelos": ORDEN_MODELOS, "brazos": list(BRAZOS),
                     "base_heterogeneos": list(BASE), "mejor_individual": MEJOR_INDIVIDUAL,
                     "pliegues": 5, "n_bits_sortslice": N_BITS, "radio_morgan": 2,
                     "vocabulario": "se aprende en cada ajuste; los pliegues internos del stacking lo heredan",
                     "metricas_principales": list(PRINCIPALES),
                     "metrica_tercera_lectura": "auroc",
                     "afinados": "ninguno; el 05 midió que no pagan",
                     "pliegues_brecha": PLIEGUES_BRECHA,
                     "regla_mejora": {"umbral_por_metrica": UMBRALES,
                                      "minimo_evaluaciones": MINIMO_PLIEGUES, "de": 15,
                                      "criterio": "las dos metricas principales a la vez",
                                      "umbral": "mayor desviacion del control entre sus 15 evaluaciones",
                                      "comparacion_decisiva": "agregado contra descriptores",
                                      "estratos": "exploratorios",
                                      "estado": "no pre-registrada; las 15 evaluaciones no son independientes"},
                     "control_registrado": "2026-10-29-artemio-1.5-ensambles-permeabilidad",
                     "prueba_reservada": "no se toca; se lee una sola vez en el 05"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "bbb_meta.parquet",
                    PROCESADOS / "bbb_descriptores_filtrados_sin_normalizar.parquet"]) as c:
    metricas = {
        "todos": {f"{b}/{m}": v for (b, m), v in tabla_todos.to_dict(orient="index").items()},
        "comparacion_todos": {f"{b}/{m}/{k}": v for (b, m, k), v in
                              comparacion_todos.to_dict(orient="index").items()},
        "brecha": {f"{b}/{m}": v for (b, m), v in brecha.to_dict(orient="index").items()},
        "control_reproduce": {m: float(control[("diferencia", m)].abs().max())
                              for m in ("mcc", "auprc_minoritaria", "auroc")},
        **{e: {f"{b}/{m}": v for (b, m), v in tabla(res, e).to_dict(orient="index").items()}
           for e in por_origen}}
    for brazo in orden_brazos:
        _, contra, _ = contra_individual(res, brazo)
        metricas[f"contra_individual/{brazo}"] = {f"{m}/{k}": v for (m, k), v in
                                                  contra.to_dict(orient="index").items()}
    c.metricas(metricas)

print(c.carpeta.name)

2026-10-31-artemio-1.3-ensambles-permeabilidad-huellas


In [13]:
# Las 15 evaluaciones por brazo, modelo y estrato. El cuaderno lo **escribe** y no lo lee: las
# conclusiones salen de las tablas de arriba. El CSV es para rehacer la lectura y las diferencias
# pareadas sin repetir la corrida.
# Va después del registro a propósito: escribirlo antes dejaría el árbol sucio al registrar.
SALIDA = DATOS / "05c_brazos.csv"
res.to_csv(SALIDA, index=False)
print(f"{SALIDA.relative_to(RAIZ)} · {len(res):,} filas")

notebooks/datos/05c_brazos.csv · 1,080 filas


## 3.1 Lectura

*La escribe el equipo mirando las tablas de arriba, y no la decide la primera fila. Lo que hay que contestar, dicho antes de ver los resultados para que la lectura no se acomode a ellos:*

1. *¿Alguna fila de `mejora` sale verdadera? Si ninguna, decirlo es el resultado, y hay que decirlo con el umbral a la vista en vez de bajarlo.*
2. *¿Algún ensamble se comporta distinto que los individuales del 03c, es decir, saca de las huellas algo que un modelo solo no saca? La comparación es contra el 03c, no contra cero.*
3. *¿Dónde cae la diferencia por origen? Si vuelve a caer fuera de los productos naturales, como en la fase 2, eso pesa más que el promedio.*
4. *¿Qué dice la brecha? Una ganancia pequeña con una brecha grande no es una ganancia.*
5. *¿Cambia esto algo de la [1.3](../docs/decisiones.md#d1-3)? Si no, la entrada se queda como está y este cuaderno es su evidencia; si sí, hay que marcar la versión superada y decir qué la cambió.*

#### 1. Ninguna de las 27 comparaciones sale verdadera

Nueve modelos por tres métricas, con la regla cerrada antes de correr: diferencia media por encima del umbral —la desviación del control entre sus propias 15 evaluaciones, 0.081 en MCC y 0.045 en AUPRC de la minoritaria— y positiva en al menos 12 de las 15. La que más se acerca es el `boosting` en MCC, con +0.0068 ganando 11 de 15: le falta un pliegue para el mínimo y le falta un orden de magnitud para el umbral, porque 0.081 es doce veces esa diferencia. Se dice con el umbral a la vista, y no se baja.

**El control reproduce, y por eso esto se puede comparar con el 05.** Los modelos que corren con la misma representación que aquel cuaderno dan sus mismas cifras con una diferencia máxima de 4.1 × 10⁻⁵ en MCC, 4.6 × 10⁻⁵ en AUPRC de la minoritaria y 4.8 × 10⁻⁵ en AUROC. Es redondeo. La comparación es contra el 05, no contra cero.

#### 2. Los ensambles no sacan de las huellas nada que un modelo solo no saque

Las huellas suben a los que parten el espacio y bajan a los demás, y lo hacen igual sea el modelo individual o ensamble: `boosting` +0.0068 en 11 de 15, `bosque` +0.0043 en 9 y `bagging_svm` +0.0034 en 6, contra `svm_rbf` −0.0077, `stacking` −0.0165, `knn` −0.0187, `logistica` −0.0237, `votacion` −0.0263 y `mlp` −0.0302. Es el mismo reparto que el [03c](03c_Permeabilidad_huellas.ipynb) encontró sobre los individuales: una columna de Sort & Slice es la presencia o ausencia de una subestructura, que es material para un corte y ruido para una distancia o un coeficiente.

**Lo que este cuaderno añade es que las huellas reordenan qué familia de ensamble parece la mejor.** Contra el mejor individual, sin huellas el `stacking` va +0.0093 de MCC en 12 de 15 y la `votacion` +0.0075 en 9; con huellas el `stacking` cae a +0.0003 en 7 de 15 y la `votacion` a −0.0111 en 6, mientras el `boosting` sube de +0.0003 en 8 a +0.0147 en 11 y el `bosque` de −0.0021 a +0.0099. No es que las huellas suban o bajen el desempeño: es que **cambian la respuesta a qué familia de ensamble conviene**. Sin ellas mandan los heterogéneos, con ellas los de árboles. Una conclusión sobre familias de ensamble que no diga con qué representación se midió no significa nada, y esto no depende del umbral ni de la regla.

#### 3. La diferencia cae fuera de los productos naturales, otra vez

Desglosado por origen, lo que las huellas añaden en MCC:

| modelo | todos | natural exacto | análogo estereoquímico | sin equivalente natural |
|:--|--:|--:|--:|--:|
| `bagging_svm` | +0.0034 | −0.0005 | −0.0667 | **+0.0641** |
| `svm_rbf` | −0.0077 | −0.0112 | −0.0824 | +0.0489 |
| `knn` | −0.0187 | −0.0329 | −0.0430 | +0.0340 |
| `stacking` | −0.0165 | −0.0181 | −0.0769 | +0.0218 |
| `boosting` | +0.0068 | +0.0063 | −0.0061 | +0.0200 |
| `votacion` | −0.0263 | −0.0243 | **−0.0929** | +0.0070 |
| `bosque` | +0.0043 | +0.0063 | −0.0058 | +0.0039 |

En `sin equivalente natural` las huellas ayudan a ocho de los nueve, y a algunos mucho. En `natural exacto` el efecto es pequeño y de los dos signos. Y en `análogo estereoquímico` **perjudican a los nueve, sin una sola excepción**, con pérdidas de hasta 0.093. Es el mismo sitio donde cayó la ganancia en la fase 2 según la [1.3](../docs/decisiones.md#d1-3): donde el proyecto no la necesita.

**Y ese estrato es el más difícil para todos, no solo con huellas.** En la comparativa del [05](05_Modelo_Final.ipynb) queda por debajo de `natural exacto` para los once modelos, sin huellas de por medio. Hay una razón medida: el **54.4%** de sus compuestos trae al menos un centro estereogénico **sin declarar**, contra el 35.4% de `natural exacto` y el 34.8% de `sin equivalente natural`. Es donde los registros están más incompletos, y un registro incompleto cae en este estrato sin que haya ninguna diferencia estereoquímica real, porque su InChIKey completo difiere igual. Eso no explica por qué las huellas pierden aquí: dice que el estrato mide dos cosas a la vez. Por qué pierden queda abierto en el [#183](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/183), que guarda la medición por la que la explicación mecánica que parecía evidente —que las huellas son ciegas a la quiralidad— no se sostuvo. Lo que sí acompaña al resultado, y es tabla y no hipótesis, es la brecha del punto siguiente.

#### 4. La brecha dice que para siete de los nueve las huellas son capacidad y no señal

Con huellas, la brecha entre entrenamiento y validación en AUROC se ensancha en siete modelos —`logistica` +0.0609, `mlp` +0.0439, `svm_rbf` +0.0262, `votacion` +0.0261, `bagging_svm` +0.0220, `stacking` +0.0152, `knn` +0.0093— y se estrecha solo en dos: `boosting` −0.0020 y `bosque` −0.0033. **Los dos únicos modelos a los que las huellas mejoran en la mayoría de sus evaluaciones —`boosting` en 11 de 15 y `bosque` en 9— son los dos únicos cuya brecha no se ensancha.** El `bagging_svm` sube en media (+0.0034) pero solo en 6 de 15, y su brecha se ensancha en +0.0220: no es una excepción a la lectura sino un caso que la confirma. Donde hay ganancia sostenida no hay memorización añadida, y donde hay pérdida la hay: las dos lecturas apuntan a lo mismo sin haberse mirado la una a la otra.

#### 5. La decisión 1.3 se queda como está, y este cuaderno es su evidencia para los ensambles

La [1.3](../docs/decisiones.md#d1-3) dice que en la fase 1 las huellas no mejoran a nadie y que lo que queda es el argumento de explicabilidad. Eso sigue en pie: las dos subidas sostenidas que hay no pasan la regla, caen fuera de los naturales y vienen con el reordenamiento como advertencia. No hay versión superada que marcar.

Lo que sí hay que añadir a la entrada es la advertencia nueva: **en la fase 1 las huellas entran por explicabilidad, y meterlas cambia qué ensamble parece el mejor**, así que un modelo elegido sobre la matriz ancha no es comparable con uno elegido sobre la estrecha.

# Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Andamiaje del cuaderno, escrito antes de ver cualquier resultado: el armado de los dos brazos reutilizando los ensambles del 05 sin modificarlos, y las dos comparaciones pareadas | Secciones 1 a 3 (el código de apoyo vive en `src/`). La lectura de la sección 3.1 y el primer borrador de la tabla de contribución se redactaron desde la corrida registrada y desde el registro del repositorio, para que el equipo los contrastara contra `metricas.json` y los confirmara o corrigiera. `CanalizacionConGrupos` la introduce este cuaderno y no se atribuye a nadie: es lo único que el repositorio sostiene | **Lo que el equipo decidió:** medir el cruce de ensambles y huellas aunque el 03c y la [1.3](../docs/decisiones.md#d1-3) hicieran esperar que no mejore, para tener la evidencia escrita en vez de supuesta; cuáles dos brazos y por qué no cinco; dejar los afinados fuera; y la lectura de los resultados |

### Decisiones principales

1. **Medir el cruce en vez de declararlo.** La [1.3](../docs/decisiones.md#d1-3) ya dice que las huellas no entran en la fase 1, y el 03c lo midió sobre los individuales. *Alternativa descartada:* dejar la frase «no está medido para ensambles» como límite declarado en el 05, que cuesta cero pero deja una afirmación sin respaldo en un avance que sí tiene cómo respaldarla.
2. **Dos brazos, no cinco.** `descriptores` y `descriptores+huellas`. *Alternativa descartada:* repetir los cinco brazos del 03c, que multiplica la corrida para contestar tres preguntas que ya tienen respuesta en una corrida registrada.
3. **El control se compara contra la corrida de esta misma máquina.** *Alternativa descartada:* contrastarlo con la corrida de macOS que trae el 05, que mezclaría la diferencia entre matrices del [#166](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/166) con cualquier otra diferencia y dejaría sin saber si el control reproduce.

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | Escribió el cuaderno sin ejecutar, para que lo revisara y corriera otra persona, con los nueve modelos en dos brazos y la regla de mejora cerrada antes de correr. Aportó las huellas Sort & Slice y el vocabulario que se aprende dentro de cada pliegue, que es lo que usa el brazo ancho, y la canalización que permite envolver un ensamble sin cortar el paso de los grupos. Registró la corrida de control en la misma máquina, sin la cual los dos brazos no tendrían contra qué compararse |
| Artemio Santiago Padilla Robles | Encontró que la celda de control apuntaba al identificador literal de una corrida que no existe en todas las máquinas, y reescribió la prueba para que ate la existencia de la corrida nombrada en lugar de su nombre exacto: ahora extrae los identificadores del código mostrado y comprueba que cada carpeta esté registrada, con lo que un control que compararía contra nada falla en la integración continua. Ejecutó el cuaderno y registró la corrida: las veintisiete comparaciones quedan sin cumplir la regla, y las huellas solo suben a los ensambles de árboles. Escribió la lectura de la sección 3.1 |
| José Antonio Vázquez Martínez | Escribió los ensambles de regresión y el AFT censurado de la fase 2. Ese trabajo es lo que obligó a separar el módulo de ensambles por fase, y de esa separación salió `src/models/ensambles_clasificacion.py`, que es el que este cuaderno importa sin modificar. No tiene commits en esta rama porque estuvo con el cuaderno de la fase 2 y su entrada del registro en paralelo |

*Las tres líneas esperan la confirmación de cada integrante: lo que no dejó rastro en el repositorio solo lo puede añadir su autor. Y no se verifican con `git blame`, que aquí acredita cada línea de la rama a quien abrió el PR, sino con `git log --follow` sobre el archivo y con el PR que introdujo el cambio.*